In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import Lasso
from sklearn.metrics import root_mean_squared_error
import pickle

In [2]:
import mlflow
import os
db_path = os.path.abspath("/workspaces/MLOPS/03-Experiment_tracking/mlflow.db")
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment("nyc-taxi-exp")

#mlflow ui --backend-store-uri sqlite:///mlflow.db

<Experiment: artifact_location='/workspaces/MLOPS/03-Experiment_tracking/mlruns/2', creation_time=1788511561872, effective_trace_archival_retention=None, experiment_id='2', last_update_time=1788511561872, lifecycle_stage='active', name='nyc-taxi-exp', tags={}, trace_location=None, workspace='default'>

In [3]:
def read_dataframe(path):
    df=pd.read_parquet(path)

    df.lpep_dropoff_datetime = pd.to_datetime(df.lpep_dropoff_datetime)
    df.lpep_pickup_datetime = pd.to_datetime(df.lpep_pickup_datetime)
    
    df['duration'] = df.lpep_dropoff_datetime-df.lpep_pickup_datetime
    
    df.duration = df.duration.apply(lambda td: td.total_seconds() / 60)
    
    df=df[((df.duration>=1) & (df.duration<=60))]
    
    categorical = ['PULocationID', 'DOLocationID']
    
    df[categorical]=df[categorical].astype(str)

    return df

In [4]:
df_train=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet')
df_val=read_dataframe('https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet')

In [5]:
len(df_train), len(df_val)

(38088, 35319)

In [6]:
categorical = ['PULocationID', 'DOLocationID']
numerical = ['trip_distance']

dv=DictVectorizer()

train_dicts = df_train[categorical + numerical].to_dict(orient='records')
X_train = dv.fit_transform(train_dicts)

val_dicts = df_val[categorical + numerical].to_dict(orient='records')
X_val = dv.transform(val_dicts)

In [7]:
target='duration'
y_train= df_train[target].values
y_val= df_val[target].values

In [8]:
lr = LinearRegression()
lr.fit(X_train, y_train)

y_pred = lr.predict(X_val)

root_mean_squared_error(y_val, y_pred)

8.33683572243964

In [13]:
X_train.indices = X_train.indices.astype('int32')
X_train.indptr = X_train.indptr.astype('int32')

X_val.indices = X_val.indices.astype('int32')
X_val.indptr = X_val.indptr.astype('int32')

with mlflow.start_run():
    mlflow.set_tag("developer", "aitor")
    
    mlflow.log_param("train-data-path","https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-01.parquet")
    mlflow.log_param("valid-data-path","https://d37ci6vzurychx.cloudfront.net/trip-data/green_tripdata_2026-02.parquet")
    
    alpha=0.1
    mlflow.log_param("alpha", alpha)
    lr = Lasso(alpha)
    lr.fit(X_train, y_train)

    y_pred = lr.predict(X_val)

    rmse=root_mean_squared_error(y_val, y_pred)
    mlflow.log_metric("rmse", rmse)
    
    mlflow.log_artifact(local_path="models/lin_reg.bin", artifact_path="models_pickle")

In [10]:
import xgboost as xgb

from hyperopt import fmin, tpe, hp, STATUS_OK, Trials
from hyperopt.pyll import scope


train=xgb.DMatrix(X_train, label=y_train)
valid=xgb.DMatrix(X_val, label=y_val)

def objective(params):
    with mlflow.start_run():
        mlflow.set_tag("model", "xgboost")
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=1000,
            evals=[(valid, "validation")],
            early_stopping_rounds=50,
            verbose_eval=100
        )
        y_pred=booster.predict(valid)
        rmse = root_mean_squared_error(y_val, y_pred)
        mlflow.log_metric("rmse", rmse)
        
    return {'loss':rmse, 'status': STATUS_OK}

In [11]:
search_space={
    'max_depth': scope.int(hp.quniform('max_depth', 4, 100, 1)),
    'learning_rate': hp.loguniform('learning_rate', -3, 0),
    'reg_alpha': hp.loguniform('reg_alpha', -5, -1),
    'reg_lambda': hp.loguniform('reg_lambda', -6, -1),
    'min_child_weight': hp.loguniform('min_child_weight', -1, 3),
    'objective': 'reg:squarederror',
    'seed': 42,
}

best_result = fmin(fn=objective,
                   space=search_space,
                   algo=tpe.suggest,
                   max_evals=50,
                   trials=Trials(),
                   verbose=False)

[0]	validation-rmse:9.94776
[100]	validation-rmse:5.89979
[200]	validation-rmse:5.83767
[300]	validation-rmse:5.81021


KeyboardInterrupt: 

In [ ]:
with open('models/lin_reg.bin', 'wb') as f_out:
    pickle.dump((dv,lr), f_out)